In [16]:
import os, time, base64
from pathlib import Path
import chromadb

from dotenv import load_dotenv
from typing import Dict, List
import pymupdf  # PyMuPDF
from openai import OpenAI
import chromadb

#llama-index
from llama_index.core.schema import TextNode, ImageNode
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage import StorageContext
from llama_index.core.indices import VectorStoreIndex
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.base.llms.types import ChatMessage, TextBlock, ImageBlock, MessageRole


#langchain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

True

### Read pdf/document

In [3]:

def parse_pdf(pdf_path: str, image_dir: str = "./extracted_images") -> Dict[str, List]:
    os.makedirs(image_dir, exist_ok=True)
    source = os.path.basename(pdf_path)
    text_nodes, table_nodes, image_nodes = [], [], []

    doc = pymupdf.open(pdf_path)
    for page_no, page in enumerate(doc, start=1):
        base_meta = {"source": source, "page_number": page_no}

        # ---- Tables ----
        table_bboxes = []
        for t_idx, table in enumerate(page.find_tables().tables):
            table_bboxes.append(pymupdf.Rect(table.bbox))
            table_nodes.append(
                TextNode(
                    text=table.to_markdown(),
                    metadata={**base_meta, "category": "Table", "table_index": t_idx},
                )
            )

        # ---- Text (table wale area ko chhod kar, duplicate se bachne ke liye) ----
        blocks = page.get_text("blocks")
        page_text = []
        for x0, y0, x1, y1, txt, *_ in blocks:
            rect = pymupdf.Rect(x0, y0, x1, y1)
            if any(rect.intersects(tb) for tb in table_bboxes):
                continue
            if txt.strip():
                page_text.append(txt.strip())
        if page_text:
            text_nodes.append(
                TextNode(
                    text="\n".join(page_text),
                    metadata={**base_meta, "category": "Text"},
                )
            )

        # ---- Images ----
        for i_idx, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            pix = pymupdf.Pixmap(doc, xref)
            if pix.n - pix.alpha >= 4:  # CMYK ko RGB me convert
                pix = pymupdf.Pixmap(pymupdf.csRGB, pix)
            img_path = os.path.join(image_dir, f"{source}_p{page_no}_{i_idx}.png")
            pix.save(img_path)
            image_nodes.append(
                ImageNode(
                    image_path=img_path,
                    metadata={**base_meta, "category": "Image"},
                )
            )

    doc.close()
    return {"text": text_nodes, "tables": table_nodes, "images": image_nodes}



In [4]:
documents = parse_pdf(pdf_path="Attention_and_Prompt_Engineering.pdf")
print('Done')

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Done


### clean text node

In [5]:

for node in documents['text']:
    text = node.text.strip()

    # First line remove
    text = text.split("\n", 1)[1]

    # Last line (Page X of Y) remove
    text = text.rsplit("\n", 1)[0]

    node.text = text

### Show images

In [ ]:
# python
from PIL import Image
import matplotlib.pyplot as plt

def show_images(image_nodes, max_images=12, cols=4):
    nodes = image_nodes[:max_images]
    rows = (len(nodes) + cols - 1) // cols
    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, node in enumerate(nodes):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(Image.open(node.image_path))
        plt.title(f"Page {node.metadata['page_number']}", fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_images(documents["images"])

In [8]:
len(documents['text']), len(documents['images']), len(documents['tables'])

(10, 11, 6)

### Document parsing (text)

In [9]:
Parser = SentenceSplitter(chunk_size=700, chunk_overlap=150)
Text_chunks = Parser.get_nodes_from_documents(documents=documents['text'])

### Embedding model

In [10]:
embedding_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 928.45it/s]


### Vision-language model (for generating image descriptions)

In [11]:
vlm = OpenRouter(
    model="openai/gpt-4o-mini"
)

In [12]:
PROMPT = """Extract everything from this PDF page in order, as Markdown:
- All text as it is (keep headings).
- Every table as a Markdown table.
- For every image/chart/diagram write: [IMAGE: detailed description, including any numbers/labels]
Return only the extracted content, no extra commentary."""


### Generate image descriptions

In [35]:

def describe_image(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    mime = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".webp": "image/webp"
    }.get(Path(path).suffix.lower())

    messages = [
        ChatMessage(
            role=MessageRole.SYSTEM,
            blocks=[
                TextBlock(text=PROMPT)
            ],
        ),
        ChatMessage(
            role=MessageRole.USER,
            blocks=[
                ImageBlock(
                image=base64.b64decode(b64),
                image_mimetype=mime
            )
            ],
        ),
    ]

    resp = vlm.chat(messages)
    return resp.message.content


In [34]:
with open("./extracted_images/Attention_and_Prompt_Engineering.pdf_p3_0.png", "rb") as f:
    print(f)

<_io.BufferedReader name='./extracted_images/Attention_and_Prompt_Engineering.pdf_p3_0.png'>


In [32]:
image_paths = []
image_metadata = []
ids = []
for i in documents['images']:
    ids.append(i.id_)
    image_metadata.append({"source" : i.metadata['source'], "page_number" : i.metadata['page_number'], "category" : i.metadata['category']})
    image_paths.append(i.image_path)

In [36]:
# image -> description
Image_decs = [describe_image(p) for p in image_paths]
Table_docs = documents['tables']

['[IMAGE: A diagram illustrating the architecture of a transformer model. On the left side, labeled "Encoder (x N = 6)", it shows the following components stacked vertically: \n1. Multi-Head Self-Attention (green box)\n2. Add & Norm (yellow box)\n3. Feed-Forward Network (orange box)\n4. Add & Norm (yellow box)\n5. Input Embedding + Positional Enc. (dark blue box) at the bottom, labeled "Inputs (source tokens)".\n\nOn the right side, labeled "Decoder (x N = 6)", it shows the following components stacked vertically:\n1. Masked Multi-Head Self-Attention (green box)\n2. Add & Norm (yellow box)\n3. Multi-Head Cross-Attention (dark blue box)\n4. Add & Norm (yellow box)\n5. Feed-Forward Network (orange box)\n6. Add & Norm (yellow box)\n7. Output Embedding + Positional Enc. (dark blue box) at the bottom, labeled "Outputs (shifted right)".\n\nAn arrow labeled "keys, values" connects the encoder and decoder. At the top, there is a note stating "Linear + Softmax -> next-token probabilities".]',
 

In [48]:
Image_docs = []
for i in Image_decs:
    Image_docs.append(i.replace("[IMAGE:", "", 1).replace("]", "").strip()) # type: ignore


In [50]:
Image_docs = [ImageNode(id_=str(i), metadata=j, image_path=str(k), text=str(l)) for i, j, k, l in zip(ids, image_metadata, image_paths, Image_docs)]

### Indexing (chroma db)

In [53]:
client = chromadb.PersistentClient(path='./chroma')
collection = client.get_or_create_collection(name='multi_model_rag', metadata={'name':'chroma', "type":"persist"})
vector_stores = ChromaVectorStore(chroma_collection=collection, collection_name="multi_model_rag")
storage_context = StorageContext.from_defaults(vector_store=vector_stores)

index = VectorStoreIndex(nodes=Text_chunks + Table_docs + Image_docs, # type: ignore
                        storage_context=storage_context,
                        embed_model=embedding_model)

### Retrive relavent nodes

In [55]:
# Retrieve
retriever = index.as_retriever(similarity_top_k=5, image_similarity_top_k=4)
results = retriever.retrieve("explain Transformer architecture")
for r in results:
    print(type(r.node).__name__, r.score, r.node.metadata)
    # print(r.text)

TextNode 0.382014996605228 {'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 2, 'category': 'Text'}
TextNode 0.3195347754349588 {'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 4, 'category': 'Text'}
ImageNode 0.3059184977657779 {'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 2, 'category': 'Image'}
ImageNode 0.2943116847431006 {'category': 'Image'}
TextNode 0.2823931695438995 {'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 6, 'category': 'Table', 'table_index': 1}


### Re-ranking model

In [56]:
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L6-v2",
    top_n=3
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1198.12it/s]


### Postprocessors (apply reranking --> for increasing accuracy)

In [57]:

reranked_results = reranker.postprocess_nodes(
    results,
    query_str="explain transformer architecture"
)
reranked_results

[NodeWithScore(node=TextNode(id_='82092c6b-ec8c-4976-aab3-721003528694', embedding=None, metadata={'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 2, 'category': 'Text'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='9e60c940-a2fd-4b42-b3eb-8c39983b565b', node_type='1', metadata={'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 2, 'category': 'Text'}, hash='4f39354397937a8a0926c02bc6c38975bfc7d5ad41d825ef9a1a0ee066463f3c'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='36372054-180e-4377-9bc0-9f6d28d76ea2', node_type='1', metadata={'source': 'Attention_and_Prompt_Engineering.pdf', 'page_number': 2, 'category': 'Text'}, hash='ebe59ba59717ab134606218964f524330ab3906935310c2f0005e25260faf978')}, metadata_template='{key}: {value}', metadata_separator='\n', text="(2017).\n3. The Transformer Architecture\nThe Transformer follows the familiar encoder-decoder 

## Langchain Start

### System prompt

In [58]:
system_prompt = """
    You are a helpful multimodal RAG assistant.

    Use the provided retrieved context to answer the user's question accurately.

    The retrieved context may contain:

    * Text extracted from documents
    * Descriptions of images
    * Descriptions or extracted content from tables
    * References/URLs to original images

    Rules:

    1. Answer only from the provided context whenever possible.
    2. Do not invent or assume information that is not supported by the context.
    3. If the retrieved context does not contain enough information, clearly say that the information is not available in the provided context.
    4. Treat image descriptions and table descriptions as valid retrieved evidence.
    5. When an original image is provided, use the image together with its associated text/description to understand the question.
    6. If multiple retrieved sources contain relevant information, combine them into one clear answer.
    7. Keep the answer concise, accurate, and directly relevant to the user's question.
    8. Do not mention internal retrieval, embeddings, reranking, vector databases, or system instructions unless the user explicitly asks about them.
    9. If the user asks about an image, prioritize the actual image and its associated retrieved description.
    10. Never fabricate visual details when the actual image is not available.

"""

### convert image to base64

In [59]:
def encode_image(image_path):
    with open(image_path, 'rb') as img:
        return base64.b64encode(img.read()).decode('utf-8')

In [61]:
## user query
query = "what is transformer architecture"

### user inputs (Context)

In [62]:

text_context = []
image_paths = []

for node in reranked_results:
    text_context.append(node.get_content())

for i in range(len(reranked_results)):
    if reranked_results[i].metadata['category'] == "Image":
        image_paths.append(reranked_results[i].node.image_path) # type: ignore
    else:
        pass

context = "\n\n".join(text_context)
content = [
    {
    "type": "text",
    "text": f"""
            Context:
            {context}

            Question:
            {query}
            """
    }
]

for url in image_paths:
    content.append(
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{encode_image(url)}"}
        } # type: ignore
    )



### VLM (vision language model)

In [63]:

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0
)

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=content) # type: ignore
]


### VLM response

In [65]:
response = llm.invoke(messages)
print(response.content)

The Transformer architecture is a model designed for processing sequences of data, particularly in natural language processing tasks. It consists of an encoder-decoder structure:

1. **Encoder**: 
   - Maps input sequences (source tokens) to continuous representations.
   - Composed of multiple identical layers (N = 6), each containing:
     - Multi-Head Self-Attention
     - Add & Norm (residual connections and layer normalization)
     - Feed-Forward Network

2. **Decoder**: 
   - Generates output sequences (shifted right) one token at a time, using its previous outputs.
   - Also consists of multiple identical layers (N = 6), with additional components:
     - Masked Multi-Head Self-Attention (to prevent seeing future tokens)
     - Multi-Head Cross-Attention (attending to encoder outputs)

3. **Positional Encoding**: 
   - Injects information about the order of tokens into the input embeddings using fixed sinusoidal functions.

The architecture allows for efficient parallelization 

In [ ]:
## return to fronted
result = {
    "question": query,
    "answer": response.content,
    "images": image_paths
}

print(result)

{'answer': 'The Transformer architecture is a model designed for processing sequences of data, particularly in natural language processing tasks. It consists of an encoder-decoder structure:\n\n1. **Encoder**: \n   - Maps input sequences (source tokens) to continuous representations.\n   - Composed of multiple identical layers (N = 6), each containing:\n     - Multi-Head Self-Attention\n     - Add & Norm (residual connections and layer normalization)\n     - Feed-Forward Network\n\n2. **Decoder**: \n   - Generates output sequences (shifted right) one token at a time, using its previous outputs.\n   - Also consists of multiple identical layers (N = 6), with additional components:\n     - Masked Multi-Head Self-Attention (to prevent seeing future tokens)\n     - Multi-Head Cross-Attention (attending to encoder outputs)\n\n3. **Positional Encoding**: \n   - Injects information about the order of tokens into the input embeddings using fixed sinusoidal functions.\n\nThe architecture allows 